## THIRUMALAI K
## 212224240176

In [ ]:
%pip install groq

In [ ]:
import os
import math
import json
from groq import Groq

In [ ]:
import os
from getpass import getpass

# Read the API key from an environment variable, or prompt securely.
# The typed key is hidden and is not saved in this notebook.
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or getpass("Enter your Groq API Key: ")


In [ ]:
client = Groq(api_key=GROQ_API_KEY)
print("Groq Client initialized successfully.")

In [ ]:
models = client.models.list()
for m in models.data:
  print(m.id)

In [ ]:
def calculate_cylinder_volume(radius: float, height: float, unit: str = "cm") -> str:
    """Calculates the volume of a cylinder using formula: V = π * r² * h"""
    try:
        r = float(radius)
        h = float(height)
        if r <= 0 or h <= 0:
            return json.dumps({"error": "Radius and height must be positive numbers."})
            
        volume = math.pi * (r ** 2) * h
        
        result = {
            "radius": r,
            "height": h,
            "unit": unit,
            "volume": round(volume, 4),
            "volume_unit": f"cubic {unit} ({unit}³)",
            "formula_applied": "V = π * r^2 * h"
        }
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_cylinder_volume",
            "description": "Calculates the volume of a cylinder using its radius and height.",
            "parameters": {
                "type": "object",
                "properties": {
                    "radius": {
                        "type": "number",
                        "description": "Radius of the base circle."
                    },
                    "height": {
                        "type": "number",
                        "description": "Vertical height of the cylinder."
                    },
                    "unit": {
                        "type": "string",
                        "description": "Measurement unit (e.g., cm, m, inches)"
                    }
                },
                "required": ["radius", "height"]
            }
        }
    }
]

In [ ]:
print("Tool schema defined successfully.")

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a precise mathematical assistant. Always use the calculate_cylinder_volume tool to calculate the volume of a cylinder."
    },
    {
        "role": "user",
        "content": "What is the volume of a cylinder with radius 5 cm and height 12 cm?"
    }
]

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

In [ ]:
response_message = response.choices[0].message
print("Model Requested Tool Call:")
print(response_message.tool_calls)

In [ ]:
if response_message.tool_calls:
    
    messages.append(response_message)
    
    
    for tool_call in response_message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        print(f"\nExecuting '{func_name}' locally with arguments:", func_args)
        
        if func_name == "calculate_cylinder_volume":
            tool_output = calculate_cylinder_volume(
                radius=func_args.get("radius"),
                height=func_args.get("height"),
                unit=func_args.get("unit", "cm")
            )
            
            # 3. Append the execution output back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_output
            })

In [ ]:
final_response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
    )

In [ ]:
print("THIRUMALAI K")
print("212224240176")
import re
from IPython.display import Markdown, display

content = final_response.choices[0].message.content

# 1. Convert display math \[ ... \] or [ ... \pi ... ] to $$ ... $$
content = re.sub(
    r"\\?\[\s*(.*?\\pi.*?)\s*\\?\]", r"$$\1$$", content, flags=re.DOTALL
)

# 2. Convert inline \( ... \) to $ ... $
content = re.sub(r"\\?\(\s*([Vv])\s*\\?\)", r"$\1$", content)

# 3. Clean any leftover literal escaped brackets around math
content = content.replace(r"\[", "$$").replace(r"\]", "$$")
content = content.replace(r"\(", "$").replace(r"\)", "$")

print("\n" + "=" * 50)
print("FINAL LAB REPORT OUTPUT:")
print("=" * 50)
display(Markdown(content))